# B0 纯向量检索

In [2]:
# 用于查看当前文件运行目录
from pathlib import Path
Path.cwd()


PosixPath('/home/wzh/clinical-evidence-rag')

In [3]:
import pandas as pd

evidence_path = "./data/evidence.jsonl" # evidence 目录路径

evidences = pd.read_json(evidence_path, lines=True) # 读取 evidence.jsonl
len_evi = len(evidences) # 输出证据条目数
first_evi = evidences.iloc[0] # 输出第一条证据

print("卡片总数：", len_evi)
print("第一张 ID：", first_evi["id"])
print("第一张正文：", first_evi["text"])

卡片总数： 15
第一张 ID： LR_SCOPE
第一张正文： ACR Lung-RADS is the reporting and assessment framework for CT examinations performed as part of lung cancer screening. The examination purpose matters when selecting the framework.


In [11]:
# 接下来把十五张证据卡的正文转成向量

texts = evidences["text"].to_list() # ['ACR Lung-RAD..', '.....', ..]

# 加载英文模型，生成向量 （首次加载时，如果本地没有缓存会下载模型）
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("BAAI/bge-small-en-v1.5", device = "cuda")

# 用模型对 texts 向量化
embeddings = model.encode(
    texts,
    batch_size = 8, # 批次大小，每批处理 batch_size 个文本
    normalize_embeddings = True, # 归一化，将每个向量的 L2 范数归一化为 1
    convert_to_numpy = True # 返回 numpy 数组
)

import numpy as np

print("向量矩阵形状：", embeddings.shape) # (15, 384)
print("首个向量前 5 个数：", embeddings[0][:5]) # [-0.01424679  0.05540959 -0.0370074   0.05075302  0.0426643 ]
print("首个向量 L2 范数：", np.linalg.norm(embeddings[0])) # 1.0 # linalg 是 np 中处理线性代数的模块，norm 用于计算 L2 范数

'(MaxRetryError("HTTPSConnectionPool(host='huggingface.co', port=443): Max retries exceeded with url: /BAAI/bge-small-en-v1.5/resolve/main/modules.json (Caused by SSLError(SSLEOFError(8, '[SSL: UNEXPECTED_EOF_WHILE_READING] EOF occurred in violation of protocol (_ssl.c:1000)')))"), '(Request ID: 47c7b66d-a945-4365-91df-b2d581fbd9b5)')' thrown while requesting HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/./modules.json
Retrying in 1s [Retry 1/5].
'(MaxRetryError("HTTPSConnectionPool(host='huggingface.co', port=443): Max retries exceeded with url: /BAAI/bge-small-en-v1.5/resolve/main/modules.json (Caused by SSLError(SSLEOFError(8, '[SSL: UNEXPECTED_EOF_WHILE_READING] EOF occurred in violation of protocol (_ssl.c:1000)')))"), '(Request ID: 149de0c9-7a6b-44a8-861c-e953aefbfd12)')' thrown while requesting HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/./modules.json
Retrying in 2s [Retry 2/5].
'(MaxRetryError("HTTPSConnectionPool(host='huggingface.co', po

KeyboardInterrupt: 

In [ ]:
# 将问题编码为向量
prefix = "Represent this sentence for searching relevant passages: " # 前缀末尾最好补上一个空格
query = "Which reporting framework applies to a lung cancer screening CT?" 
"""
查询为什么要加一个前缀?
1. 模型在对比学习的过程中，会让问题向量与相关正文向量更接近，与无关正文向量更远
2. 前缀能告诉模型，这是一个搜索问题，生成一个适合查找正文的向量
3. 加前缀并不保证每道题都检索得更好，也不意味着分数会更高。
4. 前缀是选择性的，可以不加
"""

query_embedding = model.encode(
    prefix + query,
    batch_size = 1,
    normalize_embeddings = True,
    convert_to_numpy = True
)

scores = np.matmul(embeddings, query_embedding)

print("查询向量形状：", query_embedding.shape)
print("查询向量 L2 范数：", np.linalg.norm(query_embedding))
print("相似度数组形状：", scores.shape)
print("15 张卡的相似度：", scores)

查询向量形状： (384,)
查询向量 L2 范数： 1.0
相似度数组形状： (15,)
15 张卡的相似度： [0.7828288  0.6693267  0.5907548  0.5962168  0.7328152  0.6579993
 0.65181136 0.7537726  0.61952454 0.568935   0.6425054  0.65719944
 0.64110833 0.6401547  0.73510337]


In [ ]:
# Top-K：选择得分最高的 K 张卡
k = 3
ranks = np.argsort(scores) # 返回排序后各元素在原数组的位置索引，默认按得分从小到大排序 
top_indices = ranks[-k:]
top_indices = top_indices[::-1]

# 从高到底返回线索卡
for rank, idx in enumerate(top_indices, start=1):
    card = evidences.iloc[idx]
    print(f"\nTop {rank} | ID: {card['id']} | 得分: {scores[idx]:.6f}")
    print("正文：", card["text"])
    print("来源：", card["source_id"])
    print("定位：", card["locator"])


Top 1 | ID: LR_SCOPE | 得分: 0.782829
正文： ACR Lung-RADS is the reporting and assessment framework for CT examinations performed as part of lung cancer screening. The examination purpose matters when selecting the framework.
来源： acr-lungrads-scope
定位： Current Release, screening CT reporting scope

Top 2 | ID: F_SCOPE | 得分: 0.753773
正文： Fleischner 2017 addresses pulmonary nodules found incidentally on CT in adults aged 35 years or older. Lung cancer screening CT is a separate setting with its own reporting framework.
来源： fleischner-2017
定位： Introduction and Implications for Patient Care

Top 3 | ID: USP_ELIGIBILITY | 得分: 0.735103
正文： The USPSTF lung cancer screening statement addresses whether people are eligible for preventive screening. Eligibility depends on factors including age and smoking history. It does not provide a Lung-RADS category for findings from an already performed CT.
来源： uspstf-screening-2021
定位： Recommendation Summary and Patient Population Under Consideration


In [ ]:
# 将以上过程封装为函数，为之后批量评测开发题做准备
# query: 未加前缀的原始问题字符串，k: 返回几张卡，默认为 3
def retrieve(query, k=3):
    query = prefix + query
    query_embedding = model.encode(
        query,
        batch_size = 1,
        normalize_embeddings= True,
        convert_to_numpy= True
    )
    
    scores = np.matmul(embeddings, query_embedding)
    rank = np.argsort(scores)
    top_indices = rank[-k:]
    top_indices = top_indices[::-1]
    
    hits = list()
    for rank, index in enumerate(top_indices):
        card = evidences.iloc[index]
        hits.append({
            'id': card['id'],
            'score': scores[index],
            'source_id': card['source_id'],
            'locator': card['locator'],
            'text': card['text']
        })
    return hits
        
question = "Which reporting framework applies to a lung cancer screening CT?"

for check_k in (3, 1):
    hits = retrieve(question, k=check_k)
    print(f"\nk={check_k}，实际返回条数：{len(hits)}")

    for rank, hit in enumerate(hits, start=1):
        print(f"Top {rank} | ID: {hit['id']} | 得分: {hit['score']:.6f}")
        print("正文：", hit["text"])
        print("来源与定位：", hit["source_id"], "|", hit["locator"])


k=3，实际返回条数：3
Top 1 | ID: LR_SCOPE | 得分: 0.782829
正文： ACR Lung-RADS is the reporting and assessment framework for CT examinations performed as part of lung cancer screening. The examination purpose matters when selecting the framework.
来源与定位： acr-lungrads-scope | Current Release, screening CT reporting scope
Top 2 | ID: F_SCOPE | 得分: 0.753773
正文： Fleischner 2017 addresses pulmonary nodules found incidentally on CT in adults aged 35 years or older. Lung cancer screening CT is a separate setting with its own reporting framework.
来源与定位： fleischner-2017 | Introduction and Implications for Patient Care
Top 3 | ID: USP_ELIGIBILITY | 得分: 0.735103
正文： The USPSTF lung cancer screening statement addresses whether people are eligible for preventive screening. Eligibility depends on factors including age and smoking history. It does not provide a Lung-RADS category for findings from an already performed CT.
来源与定位： uspstf-screening-2021 | Recommendation Summary and Patient Population Under Consider

In [ ]:
cases = pd.read_json("./data/cases.jsonl", lines=True)
dev_cases = cases[cases['split'] == "dev"] # 筛选出所有 split 属性为 “dev” 的行
first_case = dev_cases.iloc[0]


print("题目总数：", len(cases))
print("开发题数量：", len(dev_cases))
print("开发集 split 取值：", dev_cases["split"].unique())
print("首道开发题 ID：", first_case["id"])
print("报告：", first_case["report"])
print("问题：", first_case["question"])
print("标注证据 ID：", first_case["gold_evidence_ids"])

NameError: name 'pd' is not defined

In [ ]:
report = first_case['report']
query = first_case['question']
case_query = f"Report: {report}\nQuestion: {query}" 

case_hits = retrieve(case_query, 3)

print("题目 ID：", first_case["id"])
print("检索输入：\n", case_query)
print("标注证据 ID：", first_case["gold_evidence_ids"])

for rank, hit in enumerate(case_hits, start=1):
    print(f"\nTop {rank} | ID: {hit['id']} | 得分: {hit['score']:.6f}")
    print("正文：", hit["text"])
    print("来源与定位：", hit["source_id"], "|", hit["locator"])

题目 ID： DEV-01
检索输入：
 Report: Indication: Baseline low-dose CT performed within a lung cancer screening program. History: 61-year-old adult; no known primary cancer or immunosuppression. Finding: Single solid pulmonary nodule in the right upper lobe.
Question: Which guideline should I consult for interpreting this nodule, and what fact determines that choice?
标注证据 ID： ['LR_SCOPE']

Top 1 | ID: F_SCOPE | 得分: 0.794287
正文： Fleischner 2017 addresses pulmonary nodules found incidentally on CT in adults aged 35 years or older. Lung cancer screening CT is a separate setting with its own reporting framework.
来源与定位： fleischner-2017 | Introduction and Implications for Patient Care

Top 2 | ID: AC_INCIDENTAL | 得分: 0.763256
正文： ACR Appropriateness Criteria for incidental indeterminate pulmonary nodules includes variants for adults aged 35 or older with a nodule incidentally found on chest CT. The criteria address the appropriateness of the next imaging study in a specified scenario.
来源与定位： acr-inci

In [ ]:

all_hits = retrieve(case_query, len(evidences))

print("题目：", first_case["id"])
print("候选数量：", len(all_hits))

for rank, hit in enumerate(all_hits, start=1):
    print(f"{rank:2d} | {hit['id']:16s} | {hit['score']:.6f}")

题目： DEV-01
候选数量： 15
 1 | F_SCOPE          | 0.794287
 2 | AC_INCIDENTAL    | 0.763256
 3 | AC_RADIOGRAPH    | 0.757626
 4 | USP_ELIGIBILITY  | 0.727059
 5 | AC_PARTIAL       | 0.717354
 6 | LR_SCOPE         | 0.692640
 7 | LR_NONCANCER     | 0.688513
 8 | F_RISK           | 0.678222
 9 | LR_INCOMPLETE    | 0.660480
10 | LR_MEASUREMENT   | 0.652191
11 | F_MULTIPLE       | 0.651386
12 | LR_NEGATIVE      | 0.650667
13 | LR_PRIOR         | 0.632191
14 | F_EXCLUSIONS     | 0.629995
15 | LR_MULTIPLE      | 0.609438


In [ ]:
# 用于计算召回率：Recall@K = 前 K 项中命中的标注卡数量 / 该题标注卡总数
def recall_at_k(retrieved_ids, gold_ids, k):
    total = len(set(gold_ids)) # 标注卡总数
    """
    取交集函数: set(a) & set(b)  或 set(a).intersection(b)  其中 intersection 函数 b 不需要转换成列表
    """
    goat = len(set(gold_ids).intersection(retrieved_ids[:k]))
    return goat / total



example_retrieved = ["A", "C", "B"]
example_gold = ["A", "B"]

for check_k in (1, 2, 3):
    value = recall_at_k(example_retrieved, example_gold, check_k)
    print(f"示例 Recall@{check_k}：", value)

for check_k in (3, 6):
    value = recall_at_k([x['id'] for x in all_hits], ["LR_SCOPE"], check_k)
    print(f"Recall@{check_k}: ", value)

示例 Recall@1： 0.5
示例 Recall@2： 0.5
示例 Recall@3： 1.0
Recall@3:  0.0
Recall@6:  1.0


In [ ]:
# 批量评测开发题
import time

eval_rows = []
for idx, case in dev_cases.iterrows(): # iterrows() 和 enumerate 有什么区别？# iterrows() 专门用于遍历 DataFrame，返回行索引和行内容
    case["case_id"] = case["id"]
    query = f"Report: {case['report']}\nQuestion: {case['question']}"
    time_1 = time.perf_counter()
    retrieved_results = retrieve(query, 5)
    time_2 = time.perf_counter()
    retrieved_time = (time_2 - time_1) * 1000 # 检索用时，单位为毫秒
    print(retrieved_results)
    case["retrieved_ids"] = [x['id'] for x in retrieved_results]
    case["scores"] = [x['score'] for x in retrieved_results]
    case["latency_ms"] = retrieved_time
    for check_k in (1, 3, 5):
        case[f"recall_at_{check_k}"] = recall_at_k(case["retrieved_ids"], case["gold_evidence_ids"], check_k) # 什么时候用单引号，什么时候用双引号？ # 效果相同，没有差别
    eval_rows.append(case)


eval_df = pd.DataFrame(eval_rows)

mean_recalls = eval_df[["recall_at_1", "recall_at_3", "recall_at_5"]].mean()

print("评测题数：", len(eval_df))

columns = [
    "case_id", "recall_at_1", "recall_at_3",
    "recall_at_5", "latency_ms", "retrieved_ids", "scores"
]
print("评测结果列名：", eval_df.columns.tolist())
print(eval_df[columns].to_string(index=False))

print("\n各 K 的平均召回率：")
print(mean_recalls)

[{'id': 'F_SCOPE', 'score': np.float32(0.7942873), 'source_id': 'fleischner-2017', 'locator': 'Introduction and Implications for Patient Care', 'text': 'Fleischner 2017 addresses pulmonary nodules found incidentally on CT in adults aged 35 years or older. Lung cancer screening CT is a separate setting with its own reporting framework.'}, {'id': 'AC_INCIDENTAL', 'score': np.float32(0.7632561), 'source_id': 'acr-incidental-2023', 'locator': 'Variants 2 and 3, incidental nodules on chest CT', 'text': 'ACR Appropriateness Criteria for incidental indeterminate pulmonary nodules includes variants for adults aged 35 or older with a nodule incidentally found on chest CT. The criteria address the appropriateness of the next imaging study in a specified scenario.'}, {'id': 'AC_RADIOGRAPH', 'score': np.float32(0.7576258), 'source_id': 'acr-incidental-2023', 'locator': 'Variant 1, incidental nodule on chest radiograph', 'text': 'Variant 1 addresses an incidental indeterminate pulmonary nodule firs

In [ ]:
# 保存 B0 开发集评测结果，并从文件读回检查。
import json
from pathlib import Path

import pandas as pd


def save_b0_results(result_df, evidence_df, query_prefix, *, device,
                    result_dir="results", provenance=None):
    """只负责导出已有评测结果，不重新运行模型或修改检索逻辑。"""
    recall_columns = ["recall_at_1", "recall_at_3", "recall_at_5"]
    required_columns = [
        "case_id", "split", "group", "report", "question",
        "gold_evidence_ids", "retrieved_ids", "scores", "latency_ms",
        *recall_columns,
    ]
    missing = set(required_columns) - set(result_df.columns)
    if missing:
        raise ValueError(f"评测结果缺少字段：{sorted(missing)}")
    if result_df.empty or not result_df["split"].eq("dev").all():
        raise ValueError("本次 B0 导出应包含非空的开发集结果。")
    if not result_df["case_id"].is_unique:
        raise ValueError("评测结果中出现重复题目 ID。")
    if not all(len(ids) == len(scores) == 5 for ids, scores in
               zip(result_df["retrieved_ids"], result_df["scores"])):
        raise ValueError("本轮每题应有 5 个检索 ID 和对应得分。")

    query_template = "Report: {report}\nQuestion: {question}"
    export_df = result_df.copy()
    # 将 NumPy 得分转成普通浮点数，保留列表结构。
    export_df["scores"] = export_df["scores"].map(
        lambda values: [float(value) for value in values]
    )
    export_df["retrieval_query"] = [
        query_template.format(report=report, question=question)
        for report, question in zip(export_df["report"], export_df["question"])
    ]

    result_dir = Path(result_dir).resolve()
    result_dir.mkdir(parents=True, exist_ok=True)
    results_path = result_dir / "b0_dev_results.jsonl"
    summary_path = result_dir / "b0_dev_summary.json"

    summary = {
        "stage": "B0",
        "model_name": "BAAI/bge-small-en-v1.5",
        "device": str(device),
        "normalize_embeddings": True,
        "embedding_dim": 384,
        "document_batch_size": 8,
        "query_batch_size": 1,
        "query_prefix": query_prefix,
        "query_template": query_template,
        "evidence_count": len(evidence_df),
        "case_count": len(export_df),
        "split": "dev",
        "retrieval_k": 5,
        "eval_ks": [1, 3, 5],
        "mean_recalls": export_df[recall_columns].mean().to_dict(),
        "latency_ms": {
            "mean": float(export_df["latency_ms"].mean()),
            "median": float(export_df["latency_ms"].median()),
            "measurement": "单轮逐题计时，包含首题；只计 retrieve 调用，不含模型加载和打印。",
        },
        "cases_with_incomplete_recall_at_5": export_df.loc[
            export_df["recall_at_5"] < 1, "case_id"
        ].tolist(),
        "data_note": "合成病例，人工初标，尚未经过医生复核；留出集未参与本次评测。",
        "provenance": provenance if provenance is not None else {
            "origin": "notebook_eval_df",
        },
    }

    export_df.to_json(
        results_path, orient="records", lines=True,
        force_ascii=False, double_precision=15,
    )
    with summary_path.open("w", encoding="utf-8") as file:
        json.dump(summary, file, ensure_ascii=False, indent=2, allow_nan=False)

    saved_df = pd.read_json(results_path, lines=True)
    with summary_path.open(encoding="utf-8") as file:
        loaded_summary = json.load(file)

    assert saved_df["case_id"].tolist() == export_df["case_id"].tolist()
    assert saved_df["retrieved_ids"].tolist() == export_df["retrieved_ids"].tolist()
    assert saved_df[recall_columns].mean().to_dict() == loaded_summary["mean_recalls"]
    assert all(len(ids) == len(scores) == 5 for ids, scores in
               zip(saved_df["retrieved_ids"], saved_df["scores"]))
    print("逐题结果已保存：", results_path)
    print("运行摘要已保存：", summary_path)
    print("读回题数：", len(saved_df))
    print("数据划分：", saved_df["split"].unique())
    print("首题检索 ID：", saved_df.iloc[0]["retrieved_ids"])
    print("首题得分数量：", len(saved_df.iloc[0]["scores"]))
    print("摘要中的平均召回率：", loaded_summary["mean_recalls"])
    return saved_df, loaded_summary


saved_df, loaded_summary = save_b0_results(
    eval_df, evidences, prefix, device=str(model.device),
)

    

逐题结果已保存： /home/wzh/clinical-evidence-rag/results/b0_dev_results.jsonl
运行摘要已保存： /home/wzh/clinical-evidence-rag/results/b0_dev_summary.json
读回题数： 8
数据划分： ['dev']
首题检索 ID： ['F_SCOPE', 'AC_INCIDENTAL', 'AC_RADIOGRAPH', 'USP_ELIGIBILITY', 'AC_PARTIAL']
首题得分数量： 5
摘要中的平均召回率： {'recall_at_1': 0.5625, 'recall_at_3': 0.6875, 'recall_at_5': 0.6875}
